# 🧠 TensorFlow: Regularization, Generalization & Data Augmentation
## A Comprehensive A/B Testing Guide

**Assignment Part 1 — TensorFlow Edition**

This notebook systematically explores regularization techniques, weight initialization strategies, and data augmentation methods in TensorFlow/Keras. Each technique is demonstrated with **A/B comparisons** — a baseline model vs. the regularized/augmented variant — so the impact is clearly visible.

### Table of Contents
1. **Setup & Dataset Preparation**
2. **L1 & L2 Regularization**
3. **Dropout Regularization**
4. **Early Stopping**
5. **Monte Carlo Dropout**
6. **Weight Initialization Strategies**
7. **Batch Normalization**
8. **Custom Dropout & Custom Regularization**
9. **Callbacks & TensorBoard**
10. **Keras Tuner — Hyperparameter Optimization**
11. **KerasCV Data Augmentation**
12. **Multi-Domain Data Augmentation** (Image, Text, Time Series, Tabular, Audio)

---


## 1. Setup & Dataset Preparation

In [ ]:
# ============================================================
# Install dependencies (Colab-friendly)
# ============================================================
!pip install -q keras-tuner nlpaug audiomentations tsaug albumentations

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
# ============================================================
# Load CIFAR-10 for image experiments
# ============================================================
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize to [0, 1]
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Split off a validation set
X_train, X_val = X_train_full[:40000], X_train_full[40000:]
y_train, y_val = y_train_full[:40000], y_train_full[40000:]

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")

# Quick preview
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(CLASS_NAMES[y_train[i][0]], fontsize=10)
    ax.axis('off')
plt.suptitle("CIFAR-10 Sample Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Helper: Build a configurable CNN baseline
# ============================================================
def build_cnn(input_shape=(32, 32, 3), num_classes=10,
              kernel_regularizer=None, use_dropout=False, dropout_rate=0.5,
              use_batchnorm=False, initializer='glorot_uniform'):
    """
    Flexible CNN builder for A/B testing different regularization strategies.
    """
    model = keras.Sequential(name="CNN")

    # Block 1
    model.add(layers.Conv2D(32, 3, padding='same', activation=None,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer,
                            input_shape=input_shape))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, 3, padding='same', activation=None,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(2))
    if use_dropout:
        model.add(layers.Dropout(dropout_rate))

    # Block 2
    model.add(layers.Conv2D(64, 3, padding='same', activation=None,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, 3, padding='same', activation=None,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(2))
    if use_dropout:
        model.add(layers.Dropout(dropout_rate))

    # Classifier head
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation=None,
                           kernel_initializer=initializer,
                           kernel_regularizer=kernel_regularizer))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    if use_dropout:
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model


def compile_and_train(model, epochs=30, lr=1e-3, extra_callbacks=None):
    """Compile with Adam and train, returning history."""
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    cb = extra_callbacks if extra_callbacks else []
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=epochs, batch_size=128,
                        callbacks=cb, verbose=0)
    return history


def plot_ab(hist_a, hist_b, label_a="Baseline", label_b="Regularized"):
    """Side-by-side loss & accuracy comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(hist_a.history['loss'], label=f'{label_a} train', linestyle='--')
    axes[0].plot(hist_a.history['val_loss'], label=f'{label_a} val')
    axes[0].plot(hist_b.history['loss'], label=f'{label_b} train', linestyle='--')
    axes[0].plot(hist_b.history['val_loss'], label=f'{label_b} val')
    axes[0].set_title("Loss Comparison", fontweight='bold')
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(hist_a.history['accuracy'], label=f'{label_a} train', linestyle='--')
    axes[1].plot(hist_a.history['val_accuracy'], label=f'{label_a} val')
    axes[1].plot(hist_b.history['accuracy'], label=f'{label_b} train', linestyle='--')
    axes[1].plot(hist_b.history['val_accuracy'], label=f'{label_b} val')
    axes[1].set_title("Accuracy Comparison", fontweight='bold')
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print summary
    val_loss_a = min(hist_a.history['val_loss'])
    val_loss_b = min(hist_b.history['val_loss'])
    val_acc_a = max(hist_a.history['val_accuracy'])
    val_acc_b = max(hist_b.history['val_accuracy'])
    print(f"\n{'Metric':<25} {label_a:<15} {label_b:<15}")
    print("-" * 55)
    print(f"{'Best Val Loss':<25} {val_loss_a:<15.4f} {val_loss_b:<15.4f}")
    print(f"{'Best Val Accuracy':<25} {val_acc_a:<15.4f} {val_acc_b:<15.4f}")


## 2. L1 & L2 Regularization

**Concept:** L1 (Lasso) adds the absolute value of weights to the loss, encouraging sparsity. L2 (Ridge) adds the squared weights, discouraging large values and distributing weight magnitudes more evenly.

**A/B Test:** We compare a baseline (no regularization) against L1, L2, and combined L1+L2 (Elastic Net).


In [ ]:
# ============================================================
# 2a — Baseline (no regularization)
# ============================================================
print("Training BASELINE model (no regularization)...")
baseline_model = build_cnn()
hist_baseline = compile_and_train(baseline_model, epochs=30)


In [ ]:
# ============================================================
# 2b — L2 Regularization
# ============================================================
print("Training L2 REGULARIZED model (lambda=1e-4)...")
l2_model = build_cnn(kernel_regularizer=regularizers.l2(1e-4))
hist_l2 = compile_and_train(l2_model, epochs=30)

plot_ab(hist_baseline, hist_l2, "Baseline", "L2 (1e-4)")


In [ ]:
# ============================================================
# 2c — L1 Regularization
# ============================================================
print("Training L1 REGULARIZED model (lambda=1e-5)...")
l1_model = build_cnn(kernel_regularizer=regularizers.l1(1e-5))
hist_l1 = compile_and_train(l1_model, epochs=30)

plot_ab(hist_baseline, hist_l1, "Baseline", "L1 (1e-5)")


In [ ]:
# ============================================================
# 2d — Elastic Net (L1 + L2)
# ============================================================
print("Training ELASTIC NET model (l1=1e-5, l2=1e-4)...")
elastic_model = build_cnn(kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4))
hist_elastic = compile_and_train(elastic_model, epochs=30)

plot_ab(hist_baseline, hist_elastic, "Baseline", "ElasticNet")


In [ ]:
# ============================================================
# 2e — Weight distribution analysis
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
models_list = [baseline_model, l1_model, l2_model, elastic_model]
titles = ["No Reg", "L1", "L2", "Elastic Net"]

for ax, model, title in zip(axes, models_list, titles):
    all_weights = np.concatenate([w.numpy().flatten()
                                  for w in model.trainable_weights
                                  if len(w.shape) >= 2])
    ax.hist(all_weights, bins=100, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.3)
    ax.set_title(f"{title}\nstd={all_weights.std():.4f}", fontweight='bold')
    ax.set_xlabel("Weight Value")
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle("Weight Distribution Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservation: L1 pushes many weights to exactly zero (sparsity).")
print("L2 keeps weights small but distributed. Elastic Net combines both effects.")


## 3. Dropout Regularization

**Concept:** During training, dropout randomly deactivates a fraction of neurons, forcing the network to learn redundant representations. At inference time, all neurons are active but outputs are scaled.

**A/B Test:** Baseline vs. Dropout with different rates.


In [ ]:
# ============================================================
# 3a — Dropout A/B test
# ============================================================
print("Training model WITH DROPOUT (rate=0.3)...")
dropout_model = build_cnn(use_dropout=True, dropout_rate=0.3)
hist_dropout = compile_and_train(dropout_model, epochs=30)

plot_ab(hist_baseline, hist_dropout, "No Dropout", "Dropout 0.3")


In [ ]:
# ============================================================
# 3b — Compare different dropout rates
# ============================================================
dropout_rates = [0.1, 0.3, 0.5, 0.7]
dropout_histories = {}

for rate in dropout_rates:
    print(f"Training dropout rate = {rate}...")
    m = build_cnn(use_dropout=True, dropout_rate=rate)
    h = compile_and_train(m, epochs=25)
    dropout_histories[rate] = h

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for rate, h in dropout_histories.items():
    axes[0].plot(h.history['val_loss'], label=f'rate={rate}')
    axes[1].plot(h.history['val_accuracy'], label=f'rate={rate}')

axes[0].set_title("Validation Loss by Dropout Rate", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Accuracy by Dropout Rate", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_rate = min(dropout_histories, key=lambda r: min(dropout_histories[r].history['val_loss']))
print(f"\nBest dropout rate by val loss: {best_rate}")


## 4. Early Stopping

**Concept:** Monitor validation loss during training and halt when it stops improving. This prevents the model from overfitting by training too long. The `restore_best_weights` option rolls back to the best checkpoint.

**A/B Test:** Full 60-epoch training vs. early stopping.


In [ ]:
# ============================================================
# 4a — Without early stopping (train for many epochs)
# ============================================================
print("Training for 60 epochs WITHOUT early stopping...")
long_model = build_cnn()
hist_long = compile_and_train(long_model, epochs=60)


In [ ]:
# ============================================================
# 4b — With early stopping
# ============================================================
early_stop_cb = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

print("Training WITH early stopping (patience=7)...")
es_model = build_cnn()
hist_es = compile_and_train(es_model, epochs=60, extra_callbacks=[early_stop_cb])

plot_ab(hist_long, hist_es, "No Early Stop (60ep)", "Early Stopping")

actual_epochs = len(hist_es.history['loss'])
print(f"\nEarly stopping halted at epoch: {actual_epochs}")
print(f"Saved {60 - actual_epochs} epochs of unnecessary training!")


## 5. Monte Carlo Dropout

**Concept:** At inference time, keep dropout **active** and run multiple forward passes. The variance across predictions gives an **uncertainty estimate**. This is a practical approximation to Bayesian inference.

**Use case:** Safety-critical applications (medical, autonomous driving) where knowing *how confident* the model is matters as much as the prediction itself.


In [ ]:
# ============================================================
# 5a — Build model with permanent dropout (training=True at inference)
# ============================================================
mc_model = build_cnn(use_dropout=True, dropout_rate=0.3)
mc_model.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
mc_model.fit(X_train, y_train, validation_data=(X_val, y_val),
             epochs=25, batch_size=128, verbose=0)

print("Model trained. Now running MC Dropout inference...")


In [ ]:
# ============================================================
# 5b — MC Dropout inference: multiple stochastic forward passes
# ============================================================
NUM_MC_SAMPLES = 50

# Select a batch of test images
test_batch = X_test[:200]
test_labels = y_test[:200].flatten()

# Run multiple forward passes with dropout active
mc_predictions = np.stack([
    mc_model(test_batch, training=True).numpy()
    for _ in range(NUM_MC_SAMPLES)
])  # Shape: (NUM_MC_SAMPLES, 200, 10)

# Mean prediction and uncertainty
mean_probs = mc_predictions.mean(axis=0)
predictive_entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
predicted_classes = mean_probs.argmax(axis=1)

# Standard prediction (dropout off)
standard_preds = mc_model(test_batch, training=False).numpy().argmax(axis=1)

print(f"MC Dropout accuracy:  {(predicted_classes == test_labels).mean():.4f}")
print(f"Standard accuracy:    {(standard_preds == test_labels).mean():.4f}")


In [ ]:
# ============================================================
# 5c — Visualize uncertainty
# ============================================================
# Separate correct vs incorrect predictions
correct_mask = predicted_classes == test_labels
incorrect_mask = ~correct_mask

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entropy histogram
axes[0].hist(predictive_entropy[correct_mask], bins=30, alpha=0.7,
             label='Correct', color='green', edgecolor='black', linewidth=0.3)
axes[0].hist(predictive_entropy[incorrect_mask], bins=30, alpha=0.7,
             label='Incorrect', color='red', edgecolor='black', linewidth=0.3)
axes[0].set_title("Predictive Entropy: Correct vs Incorrect", fontweight='bold')
axes[0].set_xlabel("Entropy (Uncertainty)")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Show high-uncertainty examples
sorted_idx = np.argsort(predictive_entropy)[::-1]
axes[1].set_title("Top 10 Most Uncertain Predictions", fontweight='bold')
axes[1].axis('off')

for i in range(10):
    idx = sorted_idx[i]
    inset = fig.add_axes([0.55 + (i % 5) * 0.09, 0.55 - (i // 5) * 0.45, 0.08, 0.35])
    inset.imshow(test_batch[idx])
    pred_label = CLASS_NAMES[predicted_classes[idx]]
    true_label = CLASS_NAMES[test_labels[idx]]
    color = 'green' if pred_label == true_label else 'red'
    inset.set_title(f"P:{pred_label}\nT:{true_label}", fontsize=7, color=color)
    inset.axis('off')

plt.tight_layout()
plt.show()

print("\nKey Insight: MC Dropout assigns higher uncertainty to misclassified samples.")
print("This makes it invaluable for flagging unreliable predictions in production.")


## 6. Weight Initialization Strategies

**Why it matters:** Poor initialization can cause vanishing/exploding gradients, making training slow or impossible.

| Initializer | Best For | Rationale |
|---|---|---|
| **Glorot/Xavier Uniform** | sigmoid, tanh, softmax | Balances variance for symmetric activations |
| **Glorot Normal** | General purpose | Similar to above, Gaussian variant |
| **He Normal** | ReLU, Leaky ReLU, ELU | Accounts for ReLU zeroing half the outputs |
| **He Uniform** | ReLU variants | Uniform variant of He |
| **LeCun Normal** | SELU | Designed for self-normalizing networks |
| **Orthogonal** | RNNs | Prevents gradient issues in recurrent paths |
| **Zeros / Ones** | ❌ Never for weights | Creates symmetry — neurons learn identically |


In [ ]:
# ============================================================
# 6a — Compare initializers on CIFAR-10
# ============================================================
initializers = {
    'glorot_uniform': 'glorot_uniform',
    'glorot_normal': 'glorot_normal',
    'he_normal': 'he_normal',
    'he_uniform': 'he_uniform',
    'lecun_normal': 'lecun_normal',
    'orthogonal': 'orthogonal',
}

init_histories = {}
for name, init in initializers.items():
    print(f"Training with {name} initialization...")
    m = build_cnn(initializer=init)
    h = compile_and_train(m, epochs=20)
    init_histories[name] = h

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in init_histories.items():
    axes[0].plot(h.history['val_loss'], label=name)
    axes[1].plot(h.history['val_accuracy'], label=name)

axes[0].set_title("Val Loss by Initializer", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val Loss")
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Val Accuracy by Initializer", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Accuracy")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Initializer':<20} {'Best Val Loss':<15} {'Best Val Acc':<15}")
print("-" * 50)
for name, h in init_histories.items():
    print(f"{name:<20} {min(h.history['val_loss']):<15.4f} {max(h.history['val_accuracy']):<15.4f}")


In [ ]:
# ============================================================
# 6b — Demonstrate the "dead network" problem with zeros init
# ============================================================
print("WARNING: Initializing all weights to ZEROS — neurons cannot break symmetry!\n")

zero_model = build_cnn(initializer='zeros')
hist_zero = compile_and_train(zero_model, epochs=15)

print(f"\nFinal val accuracy with zeros init: {hist_zero.history['val_accuracy'][-1]:.4f}")
print("This is approximately random chance (1/10 for 10 classes).")
print("Lesson: NEVER initialize weights to zeros — use He or Glorot instead!")


## 7. Batch Normalization

**Concept:** Normalize the activations within each mini-batch to have zero mean and unit variance, then apply learnable scale (γ) and shift (β) parameters. This stabilizes training, permits higher learning rates, and acts as a mild regularizer.

**A/B Test:** Baseline vs. BatchNorm-equipped model.


In [ ]:
# ============================================================
# 7a — BatchNorm A/B test
# ============================================================
print("Training model WITH Batch Normalization...")
bn_model = build_cnn(use_batchnorm=True)
hist_bn = compile_and_train(bn_model, epochs=30)

plot_ab(hist_baseline, hist_bn, "No BatchNorm", "With BatchNorm")


In [ ]:
# ============================================================
# 7b — BatchNorm with higher learning rate
# ============================================================
print("Training BatchNorm model with HIGHER learning rate (5e-3)...")
bn_fast_model = build_cnn(use_batchnorm=True)
hist_bn_fast = compile_and_train(bn_fast_model, epochs=30, lr=5e-3)

print("Training baseline with same high learning rate...")
baseline_fast = build_cnn()
hist_baseline_fast = compile_and_train(baseline_fast, epochs=30, lr=5e-3)

plot_ab(hist_baseline_fast, hist_bn_fast,
        "No BN (lr=5e-3)", "BN (lr=5e-3)")

print("\nBatch Normalization stabilizes training even with aggressive learning rates.")


## 8. Custom Dropout & Custom Regularization

Here we implement **custom** versions to demonstrate the inner workings and show how to extend Keras.


In [ ]:
# ============================================================
# 8a — Custom Dropout Layer (spatial + alpha dropout)
# ============================================================
class AlphaDropout(layers.Layer):
    """
    Alpha Dropout: designed for SELU activations.
    Instead of zeroing units, it sets them to the negative saturation value,
    preserving the self-normalizing property.
    """
    def __init__(self, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate
        # SELU parameters
        self.alpha = 1.6732632423543772
        self.scale = 1.0507009873554805

    def call(self, inputs, training=None):
        if not training:
            return inputs

        kept = tf.random.uniform(tf.shape(inputs)) >= self.rate
        kept = tf.cast(kept, inputs.dtype)

        # Saturation value for SELU
        sat_val = -self.alpha * self.scale

        # Replace dropped units with saturation value
        output = inputs * kept + sat_val * (1.0 - kept)

        # Affine correction to maintain mean & variance
        a = (1.0 - self.rate) * (1.0 + self.rate * sat_val ** 2)
        a = tf.math.rsqrt(a)
        b = -a * sat_val * self.rate

        return a * output + b

    def get_config(self):
        config = super().get_config()
        config.update({"rate": self.rate})
        return config

# Test it
test_layer = AlphaDropout(rate=0.2)
sample_input = tf.random.normal((4, 8))
output_train = test_layer(sample_input, training=True)
output_infer = test_layer(sample_input, training=False)
print("Alpha Dropout — training output sample:", output_train.numpy()[0, :4])
print("Alpha Dropout — inference output sample:", output_infer.numpy()[0, :4])


In [ ]:
# ============================================================
# 8b — Concrete Dropout (learnable dropout rate)
# ============================================================
class ConcreteDropout(layers.Layer):
    """
    Concrete Dropout: the dropout rate is LEARNED during training
    via the concrete (Gumbel-Softmax) relaxation trick.
    Useful when the optimal dropout rate is unknown.
    """
    def __init__(self, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature

    def build(self, input_shape):
        # Learnable logit for dropout rate
        self.p_logit = self.add_weight(
            name='p_logit',
            shape=(),
            initializer=tf.initializers.Constant(-1.0),  # starts ~sigmoid(-1)≈0.27
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs, training=None):
        if not training:
            return inputs

        p = tf.sigmoid(self.p_logit)

        # Concrete relaxation (differentiable approximation to Bernoulli)
        u = tf.random.uniform(tf.shape(inputs), minval=1e-6, maxval=1 - 1e-6)
        z = tf.sigmoid((tf.math.log(u) - tf.math.log(1.0 - u) + self.p_logit) / self.temperature)

        mask = 1.0 - z  # invert: z≈1 means drop
        return inputs * mask / (1.0 - p + 1e-6)

    @property
    def dropout_rate(self):
        return tf.sigmoid(self.p_logit).numpy()

# Demo
concrete_layer = ConcreteDropout()
_ = concrete_layer(tf.ones((2, 4)), training=True)
print(f"Initial learned dropout rate: {concrete_layer.dropout_rate:.4f}")


In [ ]:
# ============================================================
# 8c — Custom Regularizer (orthogonality regularizer)
# ============================================================
class OrthogonalRegularizer(regularizers.Regularizer):
    """
    Encourages weight matrices to be orthogonal.
    Orthogonal weights help preserve gradient norms across layers,
    reducing vanishing/exploding gradient problems.

    Penalty: ||W^T W - I||_F  (Frobenius norm of deviation from orthogonality)
    """
    def __init__(self, strength=1e-3):
        self.strength = strength

    def __call__(self, weight_matrix):
        if len(weight_matrix.shape) < 2:
            return 0.0
        # Reshape conv filters to 2D: (h*w*c_in, c_out)
        w = tf.reshape(weight_matrix, (-1, weight_matrix.shape[-1]))
        product = tf.matmul(tf.transpose(w), w)
        identity = tf.eye(tf.shape(product)[0])
        return self.strength * tf.reduce_sum(tf.square(product - identity))

    def get_config(self):
        return {"strength": self.strength}


# A/B test: standard L2 vs orthogonal regularization
print("Training with ORTHOGONAL regularizer...")
ortho_model = build_cnn(kernel_regularizer=OrthogonalRegularizer(1e-4))
hist_ortho = compile_and_train(ortho_model, epochs=25)

plot_ab(hist_baseline, hist_ortho, "Baseline", "Orthogonal Reg")


## 9. Callbacks & TensorBoard

Keras callbacks let you hook into the training loop for logging, checkpointing, learning rate scheduling, and more. TensorBoard provides interactive visualization of training metrics.


In [ ]:
# ============================================================
# 9a — Comprehensive callback setup
# ============================================================
import datetime, os

# Create log directory
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs(log_dir, exist_ok=True)

# Assemble callbacks
training_callbacks = [
    # TensorBoard logging
    callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,         # Log weight histograms every epoch
        write_graph=True,
        write_images=True,
        update_freq='epoch'
    ),

    # Save best model checkpoint
    callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),

    # Reduce learning rate on plateau
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,              # Halve the LR
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    # Early stopping as safety net
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    # CSV logger for offline analysis
    callbacks.CSVLogger('training_log.csv', append=True),
]

print("Callbacks configured:")
for cb in training_callbacks:
    print(f"  ✓ {type(cb).__name__}")


In [ ]:
# ============================================================
# 9b — Custom callback: live metric printer + gradient monitor
# ============================================================
class GradientMonitorCallback(callbacks.Callback):
    """
    Custom callback that monitors gradient magnitudes during training.
    Alerts if gradients vanish or explode.
    """
    def __init__(self, check_every=5):
        super().__init__()
        self.check_every = check_every
        self.gradient_norms = []

    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.check_every != 0:
            return

        # Compute gradients on a small batch
        batch_x = X_train[:64]
        batch_y = y_train[:64]

        with tf.GradientTape() as tape:
            preds = self.model(batch_x, training=True)
            loss = keras.losses.sparse_categorical_crossentropy(batch_y, preds)
            loss = tf.reduce_mean(loss)

        grads = tape.gradient(loss, self.model.trainable_weights)
        norms = [tf.norm(g).numpy() for g in grads if g is not None]
        avg_norm = np.mean(norms)
        self.gradient_norms.append((epoch, avg_norm))

        # Alert on extreme values
        status = ""
        if avg_norm < 1e-7:
            status = " ⚠️ VANISHING!"
        elif avg_norm > 100:
            status = " ⚠️ EXPLODING!"

        print(f"  [GradMonitor] Epoch {epoch}: avg grad norm = {avg_norm:.6f}{status}")


# Train with all callbacks
print("Training with comprehensive callbacks...\n")
callback_model = build_cnn(use_batchnorm=True, use_dropout=True, dropout_rate=0.3)
hist_cb = compile_and_train(
    callback_model,
    epochs=40,
    extra_callbacks=training_callbacks + [GradientMonitorCallback(check_every=5)]
)


In [ ]:
# ============================================================
# 9c — Launch TensorBoard (in Colab)
# ============================================================
# Uncomment below to launch TensorBoard in Colab:
# %load_ext tensorboard
# %tensorboard --logdir logs/fit

print("To visualize in Colab, uncomment the TensorBoard magic commands above.")
print(f"Logs saved to: {log_dir}")
print(f"Training CSV saved to: training_log.csv")

# Plot the training with LR schedule
import pandas as pd
if os.path.exists('training_log.csv'):
    df = pd.read_csv('training_log.csv')
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(df['loss'], label='Train Loss')
    axes[0].plot(df['val_loss'], label='Val Loss')
    axes[0].set_title("Loss Curve", fontweight='bold')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(df['accuracy'], label='Train Acc')
    axes[1].plot(df['val_accuracy'], label='Val Acc')
    axes[1].set_title("Accuracy Curve", fontweight='bold')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    if 'lr' in df.columns:
        axes[2].plot(df['lr'], color='orange')
        axes[2].set_title("Learning Rate Schedule", fontweight='bold')
        axes[2].set_ylabel("LR"); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 10. Keras Tuner — Automated Hyperparameter Optimization

**Keras Tuner** automates the search for optimal hyperparameters. We define a search space and let the tuner explore it systematically using strategies like Random Search, Bayesian Optimization, or Hyperband.


In [ ]:
# ============================================================
# 10a — Define the tunable model
# ============================================================
import keras_tuner as kt

def build_tunable_model(hp):
    """
    Model-building function where hyperparameters are defined
    as part of the search space.
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(32, 32, 3)))

    # Tunable number of conv blocks
    for i in range(hp.Int('num_conv_blocks', min_value=1, max_value=3, default=2)):
        filters = hp.Choice(f'filters_{i}', values=[32, 64, 128])
        model.add(layers.Conv2D(filters, 3, padding='same'))

        if hp.Boolean(f'use_batchnorm_{i}', default=True):
            model.add(layers.BatchNormalization())

        model.add(layers.Activation('relu'))
        model.add(layers.MaxPooling2D(2))

        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.1)
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Flatten())

    # Tunable dense layer
    dense_units = hp.Choice('dense_units', values=[64, 128, 256])
    model.add(layers.Dense(dense_units, activation='relu'))

    # Tunable regularization
    reg_type = hp.Choice('regularization', values=['none', 'l1', 'l2', 'l1l2'])
    if reg_type == 'l2':
        model.add(layers.Dense(10, activation='softmax',
                               kernel_regularizer=regularizers.l2(1e-4)))
    elif reg_type == 'l1':
        model.add(layers.Dense(10, activation='softmax',
                               kernel_regularizer=regularizers.l1(1e-5)))
    elif reg_type == 'l1l2':
        model.add(layers.Dense(10, activation='softmax',
                               kernel_regularizer=regularizers.l1_l2(1e-5, 1e-4)))
    else:
        model.add(layers.Dense(10, activation='softmax'))

    # Tunable learning rate
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Tunable model defined with search space:")
print("  - Number of conv blocks: 1-3")
print("  - Filters per block: [32, 64, 128]")
print("  - BatchNorm: True/False per block")
print("  - Dropout: 0.0-0.5 per block")
print("  - Dense units: [64, 128, 256]")
print("  - Regularization: [none, l1, l2, l1l2]")
print("  - Learning rate: 1e-4 to 1e-2 (log scale)")


In [ ]:
# ============================================================
# 10b — Run Hyperband search
# ============================================================
tuner = kt.Hyperband(
    build_tunable_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='kt_results',
    project_name='cifar10_tuning',
    overwrite=True
)

print("Starting Hyperband search...\n")
tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=3)],
    verbose=0
)

# Results
print("\n" + "=" * 60)
print("SEARCH COMPLETE — Top 3 Configurations:")
print("=" * 60)
top_hps = tuner.get_best_hyperparameters(num_trials=3)
for i, hp in enumerate(top_hps):
    print(f"\nRank {i+1}:")
    for key, val in hp.values.items():
        print(f"  {key}: {val}")


In [ ]:
# ============================================================
# 10c — Evaluate best model
# ============================================================
best_model = tuner.get_best_models(num_models=1)[0]
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

print(f"\nBest Tuned Model — Test Accuracy: {test_acc:.4f}")
print(f"Baseline Model   — Test Accuracy: {baseline_model.evaluate(X_test, y_test, verbose=0)[1]:.4f}")
print(f"\nImprovement: {test_acc - baseline_model.evaluate(X_test, y_test, verbose=0)[1]:.4f}")


## 11. KerasCV Data Augmentation

KerasCV provides GPU-accelerated augmentation layers that integrate directly into tf.data pipelines and model architectures. These are faster than CPU-based augmentations and support batched operations.


In [ ]:
# ============================================================
# 11a — KerasCV augmentation pipeline
# ============================================================
try:
    import keras_cv
    KERAS_CV_AVAILABLE = True
    print(f"KerasCV version: {keras_cv.__version__}")
except ImportError:
    !pip install -q keras-cv
    import keras_cv
    KERAS_CV_AVAILABLE = True
    print(f"KerasCV version: {keras_cv.__version__}")


In [ ]:
# ============================================================
# 11b — Build augmentation pipeline with KerasCV
# ============================================================
if KERAS_CV_AVAILABLE:
    # KerasCV augmentation layers (work with batched tensors)
    augmentation_pipeline = keras.Sequential([
        keras_cv.layers.RandomFlip(mode="horizontal"),
        keras_cv.layers.RandAugment(value_range=(0, 255), augmentations_per_image=2, magnitude=0.3),
        keras_cv.layers.CutMix(alpha=1.0),
        keras_cv.layers.MixUp(alpha=0.2),
    ], name="keras_cv_augmentation")

    # Simpler pipeline without CutMix/MixUp (for standard classification)
    simple_aug = keras.Sequential([
        keras_cv.layers.RandomFlip(mode="horizontal"),
        keras_cv.layers.RandomRotation(factor=0.1),
        keras_cv.layers.RandAugment(value_range=(0, 1), augmentations_per_image=2, magnitude=0.3),
    ], name="simple_augmentation")

    print("KerasCV augmentation pipelines created!")
    print("\nSimple pipeline layers:")
    for layer in simple_aug.layers:
        print(f"  - {layer.name}: {type(layer).__name__}")


In [ ]:
# ============================================================
# 11c — Visualize KerasCV augmentations
# ============================================================
if KERAS_CV_AVAILABLE:
    # Pick some sample images
    sample_images = X_train[:8]
    sample_batch = tf.constant(sample_images)

    fig, axes = plt.subplots(3, 8, figsize=(16, 6))

    # Row 1: originals
    for i in range(8):
        axes[0, i].imshow(sample_images[i])
        axes[0, i].set_title("Original", fontsize=8)
        axes[0, i].axis('off')

    # Row 2 & 3: augmented versions
    for row in range(1, 3):
        augmented = simple_aug(sample_batch, training=True)
        for i in range(8):
            img = augmented[i].numpy()
            img = np.clip(img, 0, 1)
            axes[row, i].imshow(img)
            axes[row, i].set_title(f"Aug v{row}", fontsize=8)
            axes[row, i].axis('off')

    plt.suptitle("KerasCV Data Augmentation Samples", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 11d — A/B test: with vs without KerasCV augmentation
# ============================================================
if KERAS_CV_AVAILABLE:
    # Model with augmentation layer baked in
    aug_model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        # Augmentation (only active during training)
        keras_cv.layers.RandomFlip(mode="horizontal"),
        keras_cv.layers.RandAugment(value_range=(0, 1), augmentations_per_image=2, magnitude=0.3),
        # Conv backbone
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])

    aug_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

    print("Training model WITH KerasCV augmentation...")
    hist_aug = aug_model.fit(X_train, y_train,
                             validation_data=(X_val, y_val),
                             epochs=30, batch_size=128, verbose=0)

    plot_ab(hist_baseline, hist_aug, "No Augmentation", "KerasCV Augmented")


## 12. Multi-Domain Data Augmentation

Beyond images, data augmentation is critical for **text, time series, tabular data, and audio**. This section demonstrates domain-specific augmentation libraries and techniques.


### 12a. Image Augmentation — tf.image & Albumentations

In [ ]:
# ============================================================
# 12a — Image augmentation with tf.image (built-in)
# ============================================================
def tf_image_augment(image):
    """Apply a chain of tf.image augmentations."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image

# Visualize
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    axes[0, i].imshow(X_train[i])
    axes[0, i].set_title("Original", fontsize=8)
    axes[0, i].axis('off')

    aug_img = tf_image_augment(X_train[i]).numpy()
    axes[1, i].imshow(aug_img)
    axes[1, i].set_title("Augmented", fontsize=8)
    axes[1, i].axis('off')

plt.suptitle("tf.image Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 12a-2 — Image augmentation with Albumentations
# ============================================================
import albumentations as A

album_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=4, max_width=4, p=0.3),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
])

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    img = (X_train[i] * 255).astype(np.uint8)
    axes[0, i].imshow(X_train[i])
    axes[0, i].set_title("Original", fontsize=8)
    axes[0, i].axis('off')

    aug_result = album_transform(image=img)
    axes[1, i].imshow(aug_result['image'])
    axes[1, i].set_title("Albumentations", fontsize=8)
    axes[1, i].axis('off')

plt.suptitle("Albumentations Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nAlbumentations supports 60+ transforms: geometric, color, noise, dropout, etc.")


### 12b. Text Augmentation — nlpaug

In [ ]:
# ============================================================
# 12b — Text augmentation with nlpaug
# ============================================================
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac

sample_texts = [
    "The movie was absolutely fantastic and I loved every minute of it.",
    "Machine learning models require careful tuning of hyperparameters.",
    "The weather forecast predicts heavy rain throughout the weekend.",
]

# Synonym augmentation
syn_aug = naw.SynonymAug(aug_src='wordnet')

# Random character insertion
char_aug = nac.RandomCharAug(action='insert', aug_char_min=1, aug_char_max=2)

# Random word deletion
del_aug = naw.RandomWordAug(action='delete', aug_p=0.2)

print("TEXT AUGMENTATION EXAMPLES")
print("=" * 70)
for text in sample_texts:
    print(f"\nOriginal:   {text}")
    print(f"Synonym:    {syn_aug.augment(text)[0]}")
    print(f"Char Ins:   {char_aug.augment(text)[0]}")
    print(f"Word Del:   {del_aug.augment(text)[0]}")
    print("-" * 70)

print("\nText augmentation is crucial for NLP tasks with limited labeled data.")
print("Techniques: synonym swap, random insertion/deletion, back-translation, contextual word embeddings.")


### 12c. Time Series Augmentation — tsaug

In [ ]:
# ============================================================
# 12c — Time series augmentation with tsaug
# ============================================================
from tsaug import TimeWarp, Crop, Quantize, Drift, Reverse, AddNoise

# Generate synthetic time series
np.random.seed(42)
t = np.linspace(0, 4 * np.pi, 200)
ts_signal = np.sin(t) + 0.3 * np.sin(3 * t) + 0.1 * np.random.randn(len(t))
ts_signal = ts_signal.reshape(1, -1, 1)  # (batch, time, channels)

# Define augmentation pipeline
ts_augmenters = {
    "TimeWarp": TimeWarp(n_speed_change=3, max_speed_ratio=3),
    "AddNoise": AddNoise(scale=0.1),
    "Drift": Drift(max_drift=0.3),
    "Quantize": Quantize(n_levels=20),
    "Reverse": Reverse(),
}

fig, axes = plt.subplots(len(ts_augmenters) + 1, 1, figsize=(14, 10), sharex=True)
axes[0].plot(ts_signal[0, :, 0], color='black', linewidth=1.5)
axes[0].set_title("Original Time Series", fontweight='bold')
axes[0].grid(True, alpha=0.3)

for i, (name, augmenter) in enumerate(ts_augmenters.items(), 1):
    augmented = augmenter.augment(ts_signal)
    axes[i].plot(augmented[0, :, 0], linewidth=1, color=plt.cm.Set1(i / len(ts_augmenters)))
    axes[i].set_title(f"Augmented: {name}", fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Time Series Data Augmentation (tsaug)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTime series augmentation is vital for: sensor data, stock prices, medical signals.")
print("Methods: time warping, jittering, slicing, magnitude scaling, window cropping.")


### 12d. Tabular Data Augmentation — SMOTE & Noise Injection

In [ ]:
# ============================================================
# 12d — Tabular data augmentation
# ============================================================
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from collections import Counter

# Create imbalanced tabular dataset
X_tab, y_tab = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_classes=3, weights=[0.7, 0.2, 0.1], random_state=42
)

print("Original class distribution:", Counter(y_tab))

# ---- Method 1: Gaussian Noise Injection ----
def add_gaussian_noise(X, y, target_class, num_samples=100, noise_std=0.1):
    """Generate synthetic samples by adding Gaussian noise to existing ones."""
    class_indices = np.where(y == target_class)[0]
    selected = np.random.choice(class_indices, size=num_samples, replace=True)
    noise = np.random.normal(0, noise_std, (num_samples, X.shape[1]))
    new_samples = X[selected] + noise
    return new_samples

# Augment minority classes
aug_X = [X_tab.copy()]
aug_y = [y_tab.copy()]

for cls in [1, 2]:  # minority classes
    n_need = Counter(y_tab)[0] - Counter(y_tab)[cls]
    new_X = add_gaussian_noise(X_tab, y_tab, cls, num_samples=n_need)
    aug_X.append(new_X)
    aug_y.append(np.full(n_need, cls))

X_augmented = np.vstack(aug_X)
y_augmented = np.concatenate(aug_y)

print("After noise augmentation:", Counter(y_augmented))

# ---- Method 2: Mixup for tabular ----
def tabular_mixup(X, y, alpha=0.2, num_samples=200):
    """Apply mixup augmentation to tabular data."""
    idx1 = np.random.randint(0, len(X), num_samples)
    idx2 = np.random.randint(0, len(X), num_samples)
    lam = np.random.beta(alpha, alpha, num_samples).reshape(-1, 1)
    X_mix = lam * X[idx1] + (1 - lam) * X[idx2]
    # For classification, use the label of the dominant sample
    y_mix = np.where(lam.flatten() > 0.5, y[idx1], y[idx2])
    return X_mix, y_mix

X_mix, y_mix = tabular_mixup(X_tab, y_tab, num_samples=300)
print("Mixup generated samples:", Counter(y_mix))

# Visualize using PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tab)
X_aug_pca = pca.transform(X_augmented[len(X_tab):])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_tab, cmap='Set1', alpha=0.6, s=15)
axes[0].set_title("Original Data", fontweight='bold')
axes[0].legend(*scatter.legend_elements(), title="Class")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_tab, cmap='Set1', alpha=0.3, s=15, label='Original')
axes[1].scatter(X_aug_pca[:, 0], X_aug_pca[:, 1], c='orange', alpha=0.5, s=10, marker='x', label='Augmented')
axes[1].set_title("After Noise Augmentation", fontweight='bold')
axes[1].legend()

plt.suptitle("Tabular Data Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 12e. Audio/Speech Augmentation — audiomentations

In [ ]:
# ============================================================
# 12e — Audio/Speech augmentation with audiomentations
# ============================================================
from audiomentations import (
    Compose as AudioCompose,
    AddGaussianNoise,
    TimeStretch,
    PitchShift,
    Shift,
    Gain,
)

# Generate synthetic audio signal (sine wave with harmonics)
sr = 16000  # sample rate
duration = 2.0
t_audio = np.linspace(0, duration, int(sr * duration), dtype=np.float32)
audio_signal = (0.5 * np.sin(2 * np.pi * 440 * t_audio) +
                0.3 * np.sin(2 * np.pi * 880 * t_audio) +
                0.1 * np.random.randn(len(t_audio)).astype(np.float32))

# Define audio augmentation pipeline
audio_augment = AudioCompose([
    AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02, p=0.8),
    TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
    Shift(min_shift=-0.2, max_shift=0.2, p=0.5),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.5),
])

# Generate augmented versions
fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
axes[0].plot(t_audio[:3000], audio_signal[:3000], color='black', linewidth=0.5)
axes[0].set_title("Original Audio", fontweight='bold')
axes[0].grid(True, alpha=0.3)

for i in range(1, 4):
    aug_audio = audio_augment(samples=audio_signal, sample_rate=sr)
    axes[i].plot(t_audio[:3000], aug_audio[:3000], linewidth=0.5,
                 color=plt.cm.Set2(i / 4))
    axes[i].set_title(f"Augmented Version {i}", fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Audio Data Augmentation (audiomentations)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAudio augmentation methods: noise injection, time stretch, pitch shift,")
print("room impulse response, gain, frequency masking (SpecAugment), etc.")


## Summary & Key Takeaways

| Technique | Effect | When to Use |
|---|---|---|
| **L1 Regularization** | Sparse weights, feature selection | When many features are irrelevant |
| **L2 Regularization** | Small distributed weights | General purpose, prevents large weights |
| **Dropout** | Ensemble effect, prevents co-adaptation | Dense layers, large networks |
| **Early Stopping** | Halts at optimal point | Always — it's free and effective |
| **MC Dropout** | Uncertainty estimation | Safety-critical predictions |
| **He Initialization** | Proper gradient flow with ReLU | Always for ReLU-based networks |
| **Batch Normalization** | Stable training, higher LR | Deep networks, CNNs |
| **Custom Regularizers** | Task-specific constraints | When standard options aren't sufficient |
| **Data Augmentation** | Larger effective dataset | Always — especially with limited data |
| **Keras Tuner** | Automated HP search | When manual tuning is impractical |

### Multi-Domain Augmentation Summary
- **Image:** Geometric transforms, color jitter, cutout, mixup, CutMix
- **Text:** Synonym swap, random word ops, back-translation, contextual augmentation
- **Time Series:** Time warping, jittering, magnitude scaling, window cropping
- **Tabular:** SMOTE, noise injection, mixup, feature perturbation
- **Audio:** Noise, time stretch, pitch shift, SpecAugment, room simulation

---
*Notebook generated for Assignment Part 1 — TensorFlow Edition*
